# Music Information Retrieval System

A Retrieval-Augmented Generation (RAG) system built with LangChain for intelligent music data analysis and question answering.

## Architecture

1. **Data Ingestion**: Multi-source music data extraction (Spotify API, Genius API, Audio transcription)
2. **Document Processing**: Text segmentation using RecursiveCharacterTextSplitter
3. **Vector Embeddings**: Semantic representations via Google Generative AI embeddings
4. **Vector Storage**: FAISS for efficient similarity search
5. **Retrieval**: Context-aware document retrieval
6. **Generation**: LLM-powered response synthesis

## Components

- **Text Splitters**: Optimized chunking for music content
- **Embeddings**: GoogleGenerativeAIEmbeddings for semantic understanding  
- **Vector Stores**: FAISS integration for scalable search
- **LLMs**: Google Gemini for natural language generation
- **Prompt Engineering**: Structured templates for consistent outputs
- **Multi-modal Support**: Text and audio input processing

## Setup & Dependencies

Install required packages and configure API credentials.

In [ ]:
!pip install --upgrade spotipy lyricsgenius python-dotenv langchain langchain-community langchain-google-genai langchain-text-splitters faiss-cpu requests openai-whisper pydub

In [ ]:
# Import required libraries
import os
import json
import re
from dotenv import load_dotenv

# Spotify and music APIs
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import lyricsgenius
import requests

# Audio processing
import whisper
from pydub import AudioSegment

# LangChain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

In [ ]:
# Load environment variables and API keys
load_dotenv()

# API Configuration
google_api_key = os.getenv("GOOGLE_API_KEY")
spotify_client_id = os.getenv("SPOTIFY_CLIENT_ID")
spotify_client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")
genius_token = os.getenv("GENIUS_ACCESS_TOKEN")

if not google_api_key:
    raise ValueError("GOOGLE_API_KEY is required")

## Music Data Retrieval System

In [ ]:
class MusicRAGSystem:
    """
    LangChain-based RAG system for music information retrieval and analysis.
    
    Integrates multiple data sources (Spotify, Genius, audio transcription) 
    with semantic search and natural language generation capabilities.
    """
    
    def __init__(self):
        self.spotify_client = None
        self.genius_client = None
        self.whisper_model = None
        self._initialize_apis()
    
    def _initialize_apis(self):
        """Initialize external API clients"""
        if spotify_client_id and spotify_client_secret:
            try:
                credentials = SpotifyClientCredentials(
                    client_id=spotify_client_id,
                    client_secret=spotify_client_secret
                )
                self.spotify_client = spotipy.Spotify(client_credentials_manager=credentials)
            except Exception as e:
                print(f"Spotify API initialization failed: {e}")
        
        if genius_token:
            try:
                self.genius_client = lyricsgenius.Genius(genius_token)
            except Exception as e:
                print(f"Genius API initialization failed: {e}")
    
    def _load_whisper_model(self):
        """Load Whisper model for audio transcription"""
        if self.whisper_model is None:
            self.whisper_model = whisper.load_model("base")
        return self.whisper_model
    
    def _detect_query_intent(self, query):
        """Analyze query to determine optimal retrieval strategy"""
        query_lower = query.lower()
        
        intent_patterns = {
            'lyrics': [r'\blyrics\b', r'\bwords\b', r'\bsing\b', r'\btext\b'],
            'artist': [r'\bartist\b', r'\bband\b', r'\bmusician\b', r'\bsinger\b'],
            'album': [r'\balbum\b', r'\bLP\b', r'\brecord\b', r'\bdiscography\b'],
            'playlist': [r'\bplaylist\b', r'\bmix\b', r'\bcompilation\b'],
            'genre': [r'\bgenre\b', r'\bstyle\b', r'\btype of music\b'],
            'track': [r'\bsong\b', r'\btrack\b', r'\btune\b', r'\bmusic\b']
        }
        
        intent_scores = {}
        for intent, patterns in intent_patterns.items():
            score = sum(1 for pattern in patterns if re.search(pattern, query_lower))
            intent_scores[intent] = score
        
        return max(intent_scores, key=intent_scores.get) if max(intent_scores.values()) > 0 else 'track'
    
    def _clean_query_terms(self, query):
        """Extract and clean search terms from natural language query"""
        stop_phrases = [
            'what is', 'who is', 'tell me about', 'what are', 'who are',
            'lyrics of', 'lyrics for', 'words of', 'text of', 'sing',
            'artist', 'band', 'musician', 'singer', 'album', 'song', 'track',
            'the', 'a', 'an', 'by', 'from'
        ]
        
        clean_query = query.lower()
        for phrase in stop_phrases:
            clean_query = re.sub(rf'\b{phrase}\b', '', clean_query)
        
        clean_query = re.sub(r'[?!.]', '', clean_query)
        return ' '.join(clean_query.split()).strip()
    
    def retrieve_music_data(self, query):
        """
        Multi-source document retrieval based on query intent analysis
        """
        if not (self.spotify_client or self.genius_client):
            raise ValueError("No data sources configured. Check API credentials.")
        
        intent = self._detect_query_intent(query)
        search_terms = self._clean_query_terms(query)
        
        documents = []
        
        # Intent-based retrieval strategy
        if intent == 'lyrics' and self.genius_client:
            lyrics_data = self._get_lyrics(search_terms)
            if lyrics_data:
                documents.append(lyrics_data)
        
        if intent == 'artist' and self.spotify_client:
            artist_data = self._search_spotify(search_terms, 'artist')
            if artist_data:
                documents.append(artist_data)
        
        if intent in ['album', 'playlist'] and self.spotify_client:
            album_data = self._search_spotify(search_terms, intent)
            if album_data:
                documents.append(album_data)
        
        # Default track search
        if self.spotify_client:
            track_data = self._search_spotify(search_terms, 'track')
            if track_data:
                documents.append(track_data)
        
        if not documents:
            raise ValueError("No relevant documents found for query.")
        
        return '\n\n'.join(documents)
    
    def _search_spotify(self, query, search_type):
        """Execute Spotify API search and format results"""
        if not self.spotify_client:
            return None
            
        try:
            results = self.spotify_client.search(q=query, type=search_type, limit=5)
            formatted_results = []
            
            if search_type == 'track':
                for track in results['tracks']['items']:
                    track_info = f"""
Track: {track['name']}
Artist: {', '.join([artist['name'] for artist in track['artists']])}
Album: {track['album']['name']}
Release Date: {track['album']['release_date']}
Duration: {track['duration_ms']//1000//60}:{(track['duration_ms']//1000)%60:02d}
Popularity: {track['popularity']}/100
"""
                    formatted_results.append(track_info)
            
            elif search_type == 'artist':
                for artist in results['artists']['items']:
                    top_tracks = self.spotify_client.artist_top_tracks(artist['id'])
                    top_songs = [track['name'] for track in top_tracks['tracks'][:3]]
                    
                    artist_info = f"""
Artist: {artist['name']}
Genres: {', '.join(artist['genres']) or 'Not specified'}
Followers: {artist['followers']['total']:,}
Popularity: {artist['popularity']}/100
Top Songs: {', '.join(top_songs)}
"""
                    formatted_results.append(artist_info)
            
            elif search_type == 'album':
                for album in results['albums']['items']:
                    album_info = f"""
Album: {album['name']}
Artist: {', '.join([artist['name'] for artist in album['artists']])}
Release Date: {album['release_date']}
Total Tracks: {album['total_tracks']}
"""
                    formatted_results.append(album_info)
            
            return '\n'.join(formatted_results)
        
        except Exception as e:
            print(f"Spotify search error: {e}")
            return None
    
    def _get_lyrics(self, query):
        """Retrieve and format lyrics from Genius API"""
        if not self.genius_client:
            return None
            
        try:
            # Parse query for song and artist
            if ' by ' in query.lower():
                parts = query.lower().split(' by ')
                song, artist = parts[0].strip(), parts[1].strip()
            elif ' - ' in query:
                parts = query.split(' - ')
                artist, song = parts[0].strip(), parts[1].strip()
            else:
                song, artist = query, None
            
            song_obj = self.genius_client.search_song(song, artist)
            if song_obj:
                return f"""
Song: {song_obj.title}
Artist: {song_obj.artist}
Lyrics:
{song_obj.lyrics}
"""
            return None
        
        except Exception as e:
            print(f"Lyrics retrieval error: {e}")
            return None
    
    def transcribe_audio(self, audio_path):
        """Convert audio file to text using Whisper"""
        try:
            model = self._load_whisper_model()
            result = model.transcribe(audio_path)
            return result["text"]
        except Exception as e:
            print(f"Audio transcription error: {e}")
            return None
    
    def process_audio_input(self, audio_path, context=None):
        """Process audio file as document source"""
        transcript = self.transcribe_audio(audio_path)
        
        if not transcript:
            raise ValueError("Audio transcription failed")
        
        if context:
            return f"Audio Content: {transcript}\n\nContext: {context}"
        
        try:
            related_data = self.retrieve_music_data(transcript)
            return f"Audio Transcript: {transcript}\n\nRelated Information:\n{related_data}"
        except:
            return f"Audio Content: {transcript}"

# Initialize the music RAG system
music_rag = MusicRAGSystem()

def get_music_documents(query):
    """Primary interface for music document retrieval"""
    return music_rag.retrieve_music_data(query)

def process_audio_file(audio_path, context=None):
    """Process audio file as document source"""
    return music_rag.process_audio_input(audio_path, context)

## Audio Processing Utilities

In [ ]:
def convert_audio_format(input_path, output_path=None):
    """Convert audio files to formats compatible with Whisper"""
    from pydub import AudioSegment
    
    if output_path is None:
        output_path = input_path.rsplit('.', 1)[0] + '_converted.wav'
    
    try:
        audio = AudioSegment.from_file(input_path)
        audio.export(output_path, format="wav")
        return output_path
    except Exception as e:
        print(f"Audio conversion failed: {e}")
        return None

## Document Processing Pipeline

In [ ]:
def create_document_chunks(text_content):
    """
    Split text into semantically meaningful chunks using RecursiveCharacterTextSplitter.
    Optimized for music-related content with custom separators.
    """
    
    text_splitter = RecursiveCharacterTextSplitter(
        separators=[
            "\n\n", "\n", "Track:", "Artist:", "Album:", "Lyrics:",
            ".", "!", "?", ",", " ", ""
        ],
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        is_separator_regex=False
    )
    
    documents = text_splitter.create_documents([text_content])
    return documents

## Vector Embedding & Storage

In [ ]:
def create_vector_store(document_chunks, api_key):
    """
    Create FAISS vector store from document chunks using Google Generative AI embeddings.
    """
    
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/embedding-001",
        google_api_key=api_key
    )
    
    vectorstore = FAISS.from_documents(
        documents=document_chunks, 
        embedding=embeddings
    )
    
    return vectorstore

## RAG Question-Answering System

In [ ]:
def setup_rag_system(vectorstore, api_key):
    """
    Configure retriever, LLM, and prompt template for the RAG pipeline.
    """
    
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 3,
            "score_threshold": 0.7
        }
    )
    
    llm = GoogleGenerativeAI(
        model="gemini-1.5-flash",
        google_api_key=api_key,
        temperature=0.3,
        max_output_tokens=1024
    )
    
    prompt_template = PromptTemplate(
        input_variables=["context", "question"],
        template="""You are a music expert assistant. Answer questions using the provided context.

Context:
{context}

Question: {question}

Answer based on the context provided. If the context doesn't contain sufficient information, state this clearly.

Answer:"""
    )
    
    return retriever, llm, prompt_template

In [ ]:
def execute_rag_query(question, retriever, llm, prompt_template):
    """
    Execute complete RAG pipeline: retrieve relevant documents and generate response.
    """
    
    # Retrieve relevant documents
    relevant_docs = retriever.get_relevant_documents(question)
    
    # Prepare context from retrieved documents
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    
    # Format prompt with context and question
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )
    
    # Generate response using LLM
    response = llm.invoke(formatted_prompt)
    
    return response, relevant_docs

def create_rag_pipeline(query_or_audio_path, is_audio=False, context=None):
    """
    Build complete RAG pipeline from data source to question-answering capability.
    """
    
    try:
        # Step 1: Retrieve/process documents
        if is_audio:
            content = process_audio_file(query_or_audio_path, context)
        else:
            content = get_music_documents(query_or_audio_path)
        
        # Step 2: Process documents
        chunks = create_document_chunks(content)
        
        # Step 3: Create vector store
        vectorstore = create_vector_store(chunks, google_api_key)
        
        # Step 4: Setup RAG system
        retriever, llm, prompt_template = setup_rag_system(vectorstore, google_api_key)
        
        return retriever, llm, prompt_template, content
        
    except Exception as e:
        print(f"RAG pipeline creation failed: {e}")
        return None, None, None, None

def query_music_system(question, retriever, llm, prompt_template):
    """Execute question against the music RAG system"""
    response, sources = execute_rag_query(question, retriever, llm, prompt_template)
    return response, sources

## System Implementation

In [ ]:
class MusicQAInterface:
    """
    Interactive interface for the music RAG system.
    Provides methods for system initialization and query execution.
    """
    
    def __init__(self):
        self.retriever = None
        self.llm = None
        self.prompt_template = None
        self.is_initialized = False
    
    def initialize_system(self, query_or_audio_path, is_audio=False, context=None):
        """Initialize RAG system with data source"""
        components = create_rag_pipeline(query_or_audio_path, is_audio, context)
        self.retriever, self.llm, self.prompt_template, _ = components
        
        if self.retriever is not None:
            self.is_initialized = True
            return True
        return False
    
    def ask_question(self, question):
        """Execute question against initialized system"""
        if not self.is_initialized:
            raise ValueError("System not initialized. Call initialize_system() first.")
        
        response, sources = query_music_system(
            question, self.retriever, self.llm, self.prompt_template
        )
        
        return {
            'answer': response,
            'sources': len(sources),
            'source_documents': sources
        }
    
    def batch_query(self, questions):
        """Execute multiple questions efficiently"""
        if not self.is_initialized:
            raise ValueError("System not initialized.")
        
        results = []
        for question in questions:
            result = self.ask_question(question)
            results.append({
                'question': question,
                'answer': result['answer'],
                'sources': result['sources']
            })
        
        return results

# Usage example functions
def setup_music_qa_system(data_source, is_audio=False):
    """Setup complete music QA system with data source"""
    qa_system = MusicQAInterface()
    
    if qa_system.initialize_system(data_source, is_audio):
        return qa_system
    else:
        raise ValueError("Failed to initialize music QA system")

def process_music_query(data_source, question, is_audio=False):
    """One-shot query processing"""
    qa_system = setup_music_qa_system(data_source, is_audio)
    return qa_system.ask_question(question)

# System status check
if google_api_key and (spotify_client_id or genius_token):
    print("Music RAG System: Ready")
    print("Available APIs:", 
          "Spotify" if spotify_client_id else "", 
          "Genius" if genius_token else "")
else:
    print("Configuration incomplete. Check API credentials.")